Create a Snowflake session

In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


Import the json that contains the API token

In [ ]:
import json

# Choose a temp directory in the Snowflake compute environment
local_path = "/tmp/token.json"

# Download the file from the stage
session.file.get("@MINDBRIDGE.PUBLIC.SECURE_STAGE/token.json", "/tmp")

# Open the JSON file and load its content
with open(local_path, "r") as file:
    # Load the JSON content
    data = json.load(file)

# Extract the API token
api_token = data.get("api_token", "").strip()

# Now `api_token` contains the token from the JSON file
print("API Token:", api_token)  # Use the token as needed

Copy the table data in SF to stage in CSV format

In [ ]:
COPY INTO @TEST_STAGE/my_data_2.csv 
FROM (SELECT * FROM MINDBRIDGE.DATASET_2B.PREJOINED_DATA_TABLE_ENTRIES limit 10)
FILE_FORMAT = (TYPE = 'CSV' COMPRESSION=NONE);

Save the table to a tmp location

In [ ]:
# Read the file directly from the stage
df_snowpark = session.read.csv("@TEST_STAGE/2022_General_Ledger.csv")


# Convert Snowpark DataFrame to Pandas DataFrame
df_pandas = df_snowpark.to_pandas()

# Show the DataFrame
print(df_pandas.columns)

#save the dataframe to tmp folder in CSV format
df_pandas.to_csv('/tmp/2022_General_Ledger.csv', index=False)

Import third party libraries

In [ ]:
# ===============Import third-party python packages===============
import fcntl
import os
import sys
import threading
import zipfile

Unpack the zipped API SDK to the tmp folder

In [ ]:

#select the zipped package that is intended to be used
list_of_packages = ["mindbridgeapi_new"]

for pkg in list_of_packages:
    session.file.get(f"@TEST_STAGE/{pkg}.zip", os.getcwd())

# File lock class for synchronizing write access to /tmp
class FileLock:
    def __enter__(self):
        self._lock = threading.Lock()
        self._lock.acquire()
        self._fd = open('/tmp/lockfile.LOCK', 'w+')
        fcntl.lockf(self._fd, fcntl.LOCK_EX)

    def __exit__(self, type, value, traceback):
        self._fd.close()
        self._lock.release()

# Get the location of the import directory.
import_dir = os.getcwd()

# Get the path to the ZIP file and set the location to extract to.
extracted = '/tmp/python_pkg_dir'

# Extract the contents of the ZIP. This is done under the file lock
# to ensure that only one worker process unzips the contents.
with FileLock():
    for pkg in list_of_packages:
        if not os.path.isdir(extracted + f"/{pkg}"):
            zip_file_path = import_dir + f"/{pkg}.zip"
            with zipfile.ZipFile(zip_file_path, 'r') as myzip:
                myzip.extractall(extracted)

# Add path to new packages
sys.path.append(extracted)
# ================================================================

import mindbridgeapi
#st.write(mindbridgeapi.__version__)

Check connection

In [ ]:
import mindbridgeapi as mbapi

token = api_token
url = "dev.mindbridge.ai"
server = mbapi.Server(url=url, token=token)

user = server.users.get_current()
print(user.first_name)
print(user.role)
print(user.id)

Create an Organization

In [ ]:
url = "dev.mindbridge.ai"
organization_name = "Mustakim Org-6 from API new10"
token = api_token

gl_path = "2022_General_Ledger.csv"

server = mbapi.Server(url=url, token=token)

organization = mbapi.OrganizationItem(name=organization_name)
organization = server.organizations.create(organization)
print(f"Created organization {organization.name!r} (id: {organization.id})")

engagement = mbapi.EngagementItem(
    name=organization_name,
    organization_id=organization.id,
    engagement_lead_id=organization.created_user_info.id,
)
engagement = server.engagements.create(engagement)

Create an analysis

In [ ]:
from datetime import date

# analysis_periods the default is the current calendar year
analysis = mbapi.AnalysisItem(
    engagement_id=engagement.id,
    currency_code="GBP",
    analysis_periods=[
        mbapi.AnalysisPeriod(end_date=date(2022, 12, 31), start_date=date(2022, 1, 1))
    ],
)

analysis = server.analyses.create(analysis)
print(f"Created analysis {analysis.name!r} (id: {analysis.id})")


file_manager_file = mbapi.FileManagerItem(engagement_id=analysis.engagement_id)
file_manager_file = server.file_manager.upload(
    input_item=file_manager_file,
    input_file='/tmp/2022_General_Ledger.csv', #this needs to be the file that is intended for MB analysis
)

analysis_source = mbapi.AnalysisSourceItem(
    engagement_id=engagement.id,
    analysis_id=analysis.id,
    file_manager_file_id=file_manager_file.id,
    analysis_period_id=analysis.analysis_periods[0].id,
    target_workflow_state=mbapi.TargetWorkflowState.COMPLETED,
)
analysis_source = server.analysis_sources.create(analysis_source)
print(f"Created analysis_source (id: {analysis_source.id})")
print("(Column mappings are auto mapped based on column names)")

analysis = server.analyses.wait_for_analysis_sources(analysis)
print("All analysis sources are completed")
analysis = server.analyses.run(analysis)
print("Analysis has been started")
analysis = server.analyses.wait_for_analysis(analysis)
print("Analysis is ready")